# Da She's Voice Engine — GPU Accelerated
Run this notebook in Colab to offload TTS to a GPU.
The notebook starts a server and exposes it via ngrok.
Your voice-server.py on Echad will detect it automatically.

In [ ]:
# 1. Install dependencies
!pip install -q TTS flask flask-cors pyngrok
!pip install -q "transformers<4.46" "torch<2.6"

In [ ]:
# 2. Download the speaker voice sample from your server
import requests
# The King's voice sample from Echad
SPEAKER_URL = "https://qwert.crousia.com/speaker.wav"
resp = requests.get(SPEAKER_URL)
with open("speaker.wav", "wb") as f:
    f.write(resp.content)
print(f"Downloaded speaker sample: {len(resp.content)} bytes")

In [ ]:
# 3. Load XTTS v2 on GPU
from TTS.api import TTS
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=torch.cuda.is_available())
print("XTTS v2 loaded on GPU. Ready.")

In [ ]:
# 4. Start Flask server
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import tempfile, os, threading, json

app = Flask(__name__)
CORS(app)

@app.route("/health")
def health():
    return jsonify({"status": "ok", "gpu": torch.cuda.is_available()})

@app.route("/synthesize", methods=["POST"])
def synthesize():
    data = request.get_json()
    text = data.get("text", "")
    if not text:
        return jsonify({"error": "text required"}), 400
    
    fd, path = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    
    tts.tts_to_file(
        text=text,
        file_path=path,
        speaker_wav="speaker.wav",
        language="en"
    )
    
    return send_file(path, mimetype="audio/wav")

# Run in background thread
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()
print("Flask server started on port 5000")

In [ ]:
# 5. Expose via Cloudflare Tunnel
import subprocess, time, re

# Install cloudflared
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Start tunnel in background
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Wait for the URL to appear
time.sleep(5)
output = proc.stdout.read() if proc.stdout else ""
match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', output)
public_url = match.group(0) if match else "(check output below)"

print(f"\n" + "="*60)
print(f"COLAB GPU ENDPOINT: {public_url}")
print(f"="*60)
print(f"\nPaste this URL into voice-server.py config.")
print(f"")
print(f"Cloudflared output:")
print(output[-500:] if len(output) > 500 else output)

# Keep running
while True:
    time.sleep(60)